# Laboratorio 03 — Funciones Avanzadas libres con PySpark

**Semana:** 02 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica las funciones de fecha, agregaciones avanzadas y window functions aprendidas en la Actividad 03 sobre un dataset **de tu elección que tenga al menos una columna de fecha o timestamp**. Puedes reutilizar el dataset del Lab 01 o Lab 02 si ya tiene columna temporal.

Entrega esperada: este notebook completo con todas las celdas ejecutadas y las celdas markdown respondidas.

## Parte 1 — Descripción del dataset

Documenta tu dataset antes de escribir código:

1. **Nombre y fuente:** ¿Cómo se llama el dataset y de dónde lo obtuviste? (incluye URL)
2. **Dominio:** ¿Qué problema o área describe?
3. **¿Por qué lo elegiste?** ¿Qué pregunta temporal quieres responder?
4. **Columna temporal:** ¿Qué columna de fecha/timestamp tiene? ¿Qué granularidad (segundos, minutos, días...)?
5. **Columna numérica principal:** ¿Cuál es la métrica clave del dataset (ventas, duración, precio, puntuación...)?
6. **Columna categórica para particionar:** ¿Qué columna usarás en el `PARTITION BY` de las window functions?
7. **Preguntas de negocio temporales:** Lista al menos 3 preguntas que solo se puedan responder con funciones de fecha o window functions.

> Requisitp mínimo del dataset: columna de fecha + columna numérica + columna categórica, más de 5.000 filas.

**Escribe tu respuesta aquí:**

## Parte 2 — Carga y perfil técnico

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col, when, isnan, sum as spark_sum

VOL = "/Volumes/workspace/default/week_2"  # ajusta si usas un volumen distinto
ARCHIVO = "tu_archivo.csv"                 # cambia por el nombre real

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

print(f"Filas: {df_raw.count():,} | Columnas: {len(df_raw.columns)}")
df_raw.printSchema()

In [ ]:
df_raw.describe().show(truncate=False)

In [ ]:
# Nulos y vacíos
total = df_raw.count()
numeric_types = {"double", "float", "long", "integer", "short", "byte"}
nulos = df_raw.select([
    spark_sum(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df_raw.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
            (col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
]).collect()[0].asDict()

print(f"{'Columna':<35} {'Nulos':>8} {'%':>8}")
print("-" * 54)
for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    print(f"{c:<35} {n:>8,} {n/total*100:>7.1f}%")

**Observaciones del perfil:** ¿Hay nulos en la columna temporal o numérica? ¿Qué harás con ellos antes de los análisis?

## Parte 3 — Preparación de la columna temporal

Asegúrate de que la columna de fecha esté en tipo `timestamp` o `date`. Luego extrae al menos **4 componentes temporales** que tengan sentido para tu dataset.

In [ ]:
# Ajusta el nombre de la columna temporal y el formato si es necesario
COL_FECHA = "fecha"        # cambia por el nombre real
COL_NUMERICA = "valor"     # cambia por la métrica clave
COL_CATEGORIA = "categoria" # cambia por la columna categórica

df = df_raw \
    .withColumn("ts", F.to_timestamp(COL_FECHA)) \
    .withColumn("hora",        F.hour("ts")) \
    .withColumn("dia_semana",  F.dayofweek("ts")) \
    .withColumn("mes",         F.month("ts")) \
    .withColumn("anio",        F.year("ts")) \
    .withColumn("semana_anio", F.weekofyear("ts")) \
    .withColumn("mes_inicio",  F.date_trunc("month", "ts"))

df.select("ts", "hora", "dia_semana", "mes", "anio", "semana_anio").show(10)

**¿A qué granularidad tienen más sentido los análisis con este dataset?** (hora, día, semana, mes) ¿Por qué?

## Parte 4 — Agregaciones avanzadas por período

Calcula al menos **3 métricas distintas** agrupando por período temporal. Muestra la evolución y comenta si ves tendencias, estacionalidad o anomalías.

In [ ]:
# Agrupación por mes — ajusta COL_NUMERICA según tu dataset
df.groupBy("mes_inicio") \
    .agg(
        F.count("*").alias("registros"),
        F.sum(COL_NUMERICA).alias("total"),
        F.avg(COL_NUMERICA).alias("promedio"),
        F.percentile_approx(COL_NUMERICA, 0.9).alias("percentil_90")
    ) \
    .orderBy("mes_inicio") \
    .show(24, truncate=False)

**¿Qué tendencias, picos o anomalías ves en la evolución temporal?**

In [ ]:
# Agrega una segunda agrupación por otro componente temporal (día de semana, hora, etc.)
# que tenga sentido para tu dominio

**Observaciones:**

## Parte 5 — Window Function: Ranking

Usa `rank()` o `dense_rank()` para clasificar registros dentro de grupos. Define un `PARTITION BY` y `ORDER BY` que tenga sentido para tu dominio.

In [ ]:
# Ajusta la partición y el orden según tu dataset
windowSpec_rank = Window.partitionBy(COL_CATEGORIA).orderBy(F.col(COL_NUMERICA).desc())

df_ranking = df.withColumn("rank_en_grupo", F.rank().over(windowSpec_rank))

# Muestra los top 3 por grupo
df_ranking.filter(F.col("rank_en_grupo") <= 3) \
    .orderBy(COL_CATEGORIA, "rank_en_grupo") \
    .select(COL_CATEGORIA, COL_NUMERICA, "rank_en_grupo") \
    .show(30, truncate=False)

**Explica:** ¿Qué `PARTITION BY` y `ORDER BY` elegiste y por qué tiene sentido en tu dominio? ¿Qué diferencia hay entre `rank()` y `dense_rank()` en tu resultado?

## Parte 6 — Window Function: LAG y detección de variaciones

Calcula la variación respecto a la observación anterior dentro de cada partición. Identifica los casos donde la variación es más pronunciada.

In [ ]:
windowSpec_lag = Window.partitionBy(COL_CATEGORIA).orderBy("ts")

df = df \
    .withColumn("valor_anterior", F.lag(COL_NUMERICA, 1).over(windowSpec_lag)) \
    .withColumn(
        "variacion",
        F.round(F.col(COL_NUMERICA) - F.col("valor_anterior"), 2)
    )

# Top 10 variaciones más grandes
df.filter(F.col("variacion").isNotNull()) \
    .orderBy(F.col("variacion").desc()) \
    .select(COL_CATEGORIA, "ts", COL_NUMERICA, "valor_anterior", "variacion") \
    .show(10, truncate=False)

**¿Qué representan las variaciones más grandes? ¿Son anomalías, eventos esperados, o artefactos del dataset?**

## Parte 7 — Window Function: Acumulado o Media móvil

Implementa **una** de las dos opciones (la que tenga más sentido para tu dataset):

- **Opción A — Acumulado:** total acumulado de la métrica principal, ordenado por fecha, particionado por categoría
- **Opción B — Media móvil:** media de los últimos N registros (elige N con criterio de dominio)

Documenta cuál elegiste y por qué.

In [ ]:
# Opción A — Acumulado
# windowSpec_acum = Window.partitionBy(COL_CATEGORIA) \
#     .orderBy("ts") \
#     .rowsBetween(Window.unboundedPreceding, Window.currentRow)
# df = df.withColumn("acumulado", F.sum(COL_NUMERICA).over(windowSpec_acum))

# Opción B — Media móvil (últimos N registros)
# N = 7
# windowSpec_rolling = Window.partitionBy(COL_CATEGORIA) \
#     .orderBy("ts") \
#     .rowsBetween(-N + 1, 0)
# df = df.withColumn("media_movil", F.round(F.avg(COL_NUMERICA).over(windowSpec_rolling), 2))

# Descomenta la opción que elijas y ajusta los parámetros

**¿Cuál opción elegiste y por qué tiene sentido para tu dataset? Si elegiste media móvil, ¿por qué ese valor de N?**

## Parte 8 — Análisis de negocio: 3 preguntas temporales

Responde las 3 preguntas temporales que planteaste en la Parte 1. Cada respuesta debe usar al menos una función de fecha o una window function. Incluye:
- Bloque de código PySpark
- Celda markdown con la conclusión en lenguaje natural

In [ ]:
# Pregunta 1:

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3:

**Conclusión pregunta 3:**

## Parte 9 — Reflexión final

Responde en esta celda:

1. ¿Cuál window function te resultó más difícil de entender o aplicar? ¿Por qué?
2. ¿En qué se diferencia una window function de un `groupBy` para responder la misma pregunta?
3. ¿Qué hallazgo temporal te sorprendió más en tu dataset?
4. ¿Qué análisis adicional harías si tuvieras más columnas o más historia temporal?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_02/laboratorios/lab_03_funciones_avanzadas.ipynb semana_02/laboratorios/<tu-nombre>/lab_03_funciones_avanzadas.ipynb

git add semana_02/laboratorios/<tu-nombre>/lab_03_funciones_avanzadas.ipynb
git commit -m "lab: semana02 lab03 window functions <nombre-dataset> - <tu-nombre>"
git push origin develop
```